# Feature Importance in Machine Learning

Understanding **which features matter** is crucial for building trustworthy models.

In this notebook we:
1. Train a Random Forest on the **Wine dataset** (sklearn)
2. Compare two methods for measuring feature importance:
   - **Built-in (impurity-based)** feature importance from the tree ensemble
   - **Permutation importance** -- how much does performance drop when a feature is shuffled?
3. Visualize and discuss which features matter and why the two methods can disagree

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, classification_report

sns.set_style('whitegrid')
np.random.seed(42)

## 1. Load and Explore the Wine Dataset

In [ ]:
wine = load_wine()
X, y = wine.data, wine.target
feature_names = wine.feature_names

print(f'Dataset shape: {X.shape}')
print(f'Classes: {wine.target_names}')
print(f'Features: {feature_names}')
print(f'\nClass distribution: {np.bincount(y)}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')

## 2. Train a Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(f'Test Accuracy: {accuracy_score(y_test, y_pred):.3f}')
print()
print(classification_report(y_test, y_pred, target_names=wine.target_names))

## 3. Built-in (Impurity-Based) Feature Importance

Random Forests provide a built-in importance score based on the **mean decrease in impurity** (Gini) across all trees. Features used for splits that produce large decreases in impurity are considered more important.

**Caveat**: This method is biased toward high-cardinality features and features with many unique values.

In [ ]:
importances_builtin = rf.feature_importances_
sorted_idx = np.argsort(importances_builtin)

fig, ax = plt.subplots(figsize=(10, 7))
colors = sns.color_palette('viridis', len(feature_names))
ax.barh(range(len(feature_names)), importances_builtin[sorted_idx], 
        color=[colors[i] for i in sorted_idx])
ax.set_yticks(range(len(feature_names)))
ax.set_yticklabels(np.array(feature_names)[sorted_idx])
ax.set_xlabel('Importance (Mean Decrease in Impurity)', fontsize=12)
ax.set_title('Built-in Feature Importance (Random Forest)', fontsize=14)
plt.tight_layout()
plt.show()

print('Top 5 features (impurity-based):')
for i in sorted_idx[::-1][:5]:
    print(f'  {feature_names[i]}: {importances_builtin[i]:.4f}')

## 4. Permutation Importance

Permutation importance measures how much the model's score **drops** when a single feature's values are randomly shuffled. This breaks the relationship between the feature and the target.

**Advantages over impurity-based**: works with any model, computed on held-out data, less biased.

In [ ]:
perm_result = permutation_importance(
    rf, X_test, y_test, n_repeats=30, random_state=42, n_jobs=-1
)

perm_sorted_idx = perm_result.importances_mean.argsort()

fig, ax = plt.subplots(figsize=(10, 7))
ax.boxplot(
    perm_result.importances[perm_sorted_idx].T,
    vert=False,
    labels=np.array(feature_names)[perm_sorted_idx]
)
ax.set_xlabel('Decrease in Accuracy', fontsize=12)
ax.set_title('Permutation Importance (on Test Set)', fontsize=14)
ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print('Top 5 features (permutation importance):')
for i in perm_sorted_idx[::-1][:5]:
    print(f'  {feature_names[i]}: {perm_result.importances_mean[i]:.4f} +/- {perm_result.importances_std[i]:.4f}')

## 5. Side-by-Side Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Impurity-based
axes[0].barh(range(len(feature_names)), importances_builtin[sorted_idx],
             color='#3498db', alpha=0.8)
axes[0].set_yticks(range(len(feature_names)))
axes[0].set_yticklabels(np.array(feature_names)[sorted_idx])
axes[0].set_xlabel('Importance')
axes[0].set_title('Impurity-Based', fontsize=13)

# Permutation-based
axes[1].barh(range(len(feature_names)), 
             perm_result.importances_mean[perm_sorted_idx],
             xerr=perm_result.importances_std[perm_sorted_idx],
             color='#e74c3c', alpha=0.8)
axes[1].set_yticks(range(len(feature_names)))
axes[1].set_yticklabels(np.array(feature_names)[perm_sorted_idx])
axes[1].set_xlabel('Decrease in Accuracy')
axes[1].set_title('Permutation-Based', fontsize=13)

fig.suptitle('Feature Importance: Two Methods Compared', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## 6. Rank Correlation Between Methods

In [ ]:
from scipy.stats import spearmanr

corr, pval = spearmanr(
    importances_builtin, 
    perm_result.importances_mean
)
print(f'Spearman rank correlation: {corr:.3f} (p-value: {pval:.4f})')

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(importances_builtin, perm_result.importances_mean, 
           s=80, color='#2ecc71', edgecolors='black', linewidth=0.5)

for i, name in enumerate(feature_names):
    ax.annotate(name, (importances_builtin[i], perm_result.importances_mean[i]),
                fontsize=8, ha='left', va='bottom')

ax.set_xlabel('Impurity-Based Importance', fontsize=12)
ax.set_ylabel('Permutation Importance', fontsize=12)
ax.set_title(f'Importance Methods Correlation (Spearman r={corr:.2f})', fontsize=13)
plt.tight_layout()
plt.show()

## Discussion

### Which features matter?
- Features like **flavanoids**, **color_intensity**, **proline**, and **alcohol** consistently rank high in both methods. These are chemically meaningful for distinguishing wine cultivars.
- **OD280/OD315 of diluted wines** (a spectroscopic measure of protein content) is also important.

### Why do the methods sometimes disagree?
- **Impurity-based importance** is computed during training and can be biased toward features with more unique values or higher cardinality.
- **Permutation importance** is computed on the test set, so it reflects how useful the feature is for generalization.
- Features that are correlated can cause issues for both methods: if two features carry similar information, removing one may not hurt much (permutation), but both may get split on often (impurity).

### Best practice
Use **permutation importance on a held-out test set** as the primary metric, and cross-check with impurity-based importance. For deeper analysis, use SHAP values (covered in the next notebook).